# Multi-Period Planning

Up to T6 every model spanned a single planning window — a day, a week — and `Sizing` was the investment tool. Real planning problems span years: maybe demand grows, maybe fuel prices ramp, maybe carbon prices kick in. `periods=[...]` adds a horizon dimension and lets each period have its own dispatch.

You'll meet:

- `periods=[...]` — extra axis on every variable. Each period shares the timestep structure but has independent dispatch.
- `period_weights=[...]` — how much each period contributes to the objective. Default: inferred from period gaps (5-year gaps → weight 5 each). Override for NPV discounting.
- **`Sizing` shifts meaning** — in multi-period mode, the optimizer picks one size *per period independently*, with `effects_per_size` charged in each period.

That last point is the key change. In a single-period model, `Sizing` represented the investment decision. In multi-period, the same `Sizing` object is per-period — natural for things you re-decide each year:

- **Grid connection fee** — annual contracted capacity with a utility (kW × €/kW/year).
- **Short-term leases** — renting equipment by the period.
- **Annual tariff brackets** — capacity-based fees committed yearly.

For a *one-time built* asset (real CAPEX paid once, not an annualized rate), use `Investment` — T8 covers that explicitly.


In [ ]:
from datetime import datetime, timedelta

import pandas as pd

from fluxopt import Carrier, Converter, Effect, Flow, Port, Sizing, optimize

## 1. Story: a bakery, three periods, rising gas prices

A bakery runs a heat-driven workshop on natural gas. The plant operations team is locking in a 15-year planning view (2025, 2030, 2035) and gas isn't standing still: a regional carbon tax is set to push prices up roughly 2× every five years (€0.08 → €0.16 → €0.30 per kWh).

Same gas → boiler → workshop heat system as T6, with three planning periods at five-year gaps. The 24-hour demand profile is the same in every period (the bakery's day doesn't change), but the fuel cost varies.


In [ ]:
import xarray as xr

n = 24
timesteps = [datetime(2025, 1, 15) + timedelta(hours=h) for h in range(n)]
periods = [2025, 2030, 2035]
demand = [10, 10, 8, 8, 8, 12, 25, 60, 70, 75, 75, 70, 65, 60, 55, 50, 45, 40, 30, 25, 20, 15, 12, 10]
peak = max(demand)

# Gas price ramps with the carbon tax — cheap today, painful by 2035.
gas_price = xr.DataArray([0.08, 0.16, 0.30], dims=['period'], coords={'period': periods})

In [ ]:
def build(**extra):
    return optimize(
        timesteps=timesteps,
        periods=periods,
        carriers=[Carrier('gas'), Carrier('heat')],
        effects=[Effect('cost', unit='EUR')],
        ports=[
            Port(
                'gas_grid',
                imports=[Flow('gas', size=1000, effects_per_flow_hour={'cost': gas_price})],
            ),
            Port(
                'workshop',
                exports=[Flow('heat', size=peak, fixed_relative_profile=[d / peak for d in demand])],
            ),
        ],
        converters=[
            Converter.boiler(
                'boiler',
                thermal_efficiency=0.9,
                fuel_flow=Flow('gas', size=300),
                thermal_flow=Flow('heat', size=Sizing(min_size=20, max_size=200, effects_per_size={'cost': 1500})),
            ),
        ],
        objective_effects='cost',
        **extra,
    )


result = build()
result.effect_totals.to_dataframe(name='total').round(2)

`effect_totals` is now 2-D — `effect × period`. Per-period cost climbs with the gas price: 2035 is roughly 4× as expensive as 2025 in operating cost.

The boiler's chosen size has a period axis — one size per period:


In [ ]:
result.solution['flow--size'].sel(flow='boiler(heat)').to_dataframe(name='size (kW)')

Each period is sized independently. Demand is identical, so the optimal size is identical, and `effects_per_size=1500` is charged in *each* period. If demand were growing, sizes would scale per period — that's the per-period nature of `Sizing` in multi-period mode.


In [ ]:
# Default weights: inferred from period gaps. 5-year gaps → weight 5 per period.
pd.Series([5, 5, 5], index=pd.Index(periods, name='period'), name='weight (default)')

In [ ]:
print(f'Objective: {result.objective:.2f} EUR  (= sum of weight × per-period cost)')

## 2. NPV weights

Future euros are worth less today. Discount each period to present value with a 5 % rate over the five-year span. Override the global weights via `period_weights=`.


In [ ]:
r = 0.05
period_offsets = [0, 5, 10]  # year offsets of each period from today
npv_weights = [round(sum(1 / (1 + r) ** (y0 + y) for y in range(5)), 3) for y0 in period_offsets]

result_npv = build(period_weights=npv_weights)

pd.DataFrame(
    {
        'flat': [5, 5, 5],
        'NPV (r=5%)': npv_weights,
        'cost/period': result_npv.effect_totals.sel(effect='cost').values.round(2),
    },
    index=pd.Index(periods, name='period'),
)

In [ ]:
pd.Series(
    {
        'Flat (default)': round(result.objective, 2),
        'NPV (r=5%)': round(result_npv.objective, 2),
    },
    name='objective (EUR)',
)

NPV-weighted is lower because future periods are discounted. Per-period sizes don't change here — demand is identical in every period — but the *objective* is shaped by how much each period counts.

Per-effect weights — say physical effects (CO₂, fuel kWh) keep flat weights while only `cost` is discounted — are configured on each `Effect` directly via `period_weights=`. See the API reference.

## Recap

`periods=[...]` adds a period axis to every variable. `period_weights=` (global on `optimize()`, or per-`Effect`) controls each period's contribution to the objective.

`Sizing` in multi-period decides capacity *per period independently*, charging `effects_per_size` each period — a natural fit for annualized cost models. When you need a single capacity choice that spans the horizon, with one-time CAPEX paid in a chosen build period, that's `Investment` — next.
